# モデルを保存・読み込みする、Hub にアップロードする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_transformers/save_load_model.ipynb)

NLP クックブック 11-13 ／ Transformers の基本操作

ファインチューニングしたモデルをファイルに保存し、あとで読み込んで使いたい。Hugging Face Hub で共有したい。

In [1]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer

name = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, id2label={0: "negative", 1: "neutral", 2: "positive"})

model.save_pretrained("my-model")       # 重み (model.safetensors) と設定 (config.json)
tokenizer.save_pretrained("my-model")   # トークナイザも必ず同じ場所に保存する
for f in sorted(os.listdir("my-model")):
    print(f"{os.path.getsize('my-model/' + f) / 1e6:8.2f} MB  {f}")

    0.00 MB  config.json
  541.32 MB  model.safetensors
    2.92 MB  tokenizer.json
    0.00 MB  tokenizer_config.json


In [2]:
from transformers import pipeline

clf = pipeline("text-classification", model="my-model")   # ディレクトリ名を渡すだけで読み込める
print(clf("これはテストです。"))                           # (未学習の分類ヘッドなので予測はでたらめ)
print(clf.model.config.id2label)

[{'label': 'negative', 'score': 0.35106903314590454}]
{0: 'negative', 1: 'neutral', 2: 'positive'}


In [3]:
# Hub にアップロードする場合 (書き込み権限のあるトークンが必要)
# from huggingface_hub import login
# login()                                          # または Colab のシークレットに HF_TOKEN を登録
# model.push_to_hub("ユーザー名/my-model", private=True)
# tokenizer.push_to_hub("ユーザー名/my-model", private=True)

- `id2label` を設定しておくと、推論結果が `LABEL_0` ではなく意味のあるラベル名で返る。
- Colab のランタイムは切断されるとファイルが消える。Drive にコピーするか Hub に上げておく。
- 重みは safetensors 形式で保存される。古い `pytorch_model.bin` (pickle) と違い、読み込み時に任意コードが実行される心配がない。